# 🚀 Regelbasierte OCR-Nachbearbeitung

## Hinweise zur Ausführung des Notebooks
Dieses **Notebook** kann auf unterschiedlichen Levels erarbeitet werden (siehe Abschnitt [„Technische Voraussetzungen“](../introduction/introduction_requirements)): 
1. Book-Only Mode
2. Cloud Mode: Dafür auf 🚀 klicken und z.B. in Colab ausführen.
3. Local Mode: Dafür auf Herunterladen ↓ klicken und „.ipynb“ wählen. 

## Übersicht
In diesem Notebook wird eine regelbasierte OCR-Nachkorrektur entwickelt und angewendet. Ziel ist es, typische Fehler, die beim OCR-Prozess historischer Texte entstehen, automatisch zu beheben und die Verbesserung messbar zu machen.

Dafür werden folgende Schritte durchgeführt:
1. Identifikation typischer OCR-Fehler im Korpus (z.B. `<` statt `ch`, `fie` statt `sie`, `ſ` statt `s`)
2. Implementierung von Korrekturregeln mit regulären Ausdrücken
3. Anwendung der Regeln auf ein Beispielbild mittels Tesseract-OCR
4. Messung der Verbesserung anhand von Precision, Recall und F1-Score im Vergleich mit einem Ground-Truth-Text
5. (Advanced) Anwendung der Korrekturregeln auf das gesamte Korpus

In [ ]:
# 🚀 Install libraries
import sys
if 'google.colab' in sys.modules:
    !sudo apt install tesseract-ocr
    !sudo apt install tesseract-ocr-frk
!pip install pytesseract pillow Levenshtein

In [1]:
import re
import pytesseract
from PIL import Image
from pathlib import Path
from tqdm import tqdm

## Typische Fehler

Im Folgenden listen wir einige typische Fehler in unserem Korpus auf:

* „fie“ statt „sie“ (Bild und Ergebnis später hinzufügen)
* „vm“, „vnd“ statt „um“, „und“
* „<“ statt „ch“

Einige Dinge sind keine Fehler, sondern Merkmale der historischen Orthographie, die wir für die weitere Verarbeitung mit modernen NLP-Tools normalisieren möchten:

* „ſ“ statt „s“

In vielen Fällen können wir dies mit einigen regulären Such- und Ersetzungsmustern beheben (z.B. jedes `<`, das nicht von Leerzeichen umgeben ist, in `ch` umwandeln).

Der Standardweg, solche Muster auf einem Computer auszudrücken und zu implementieren, sind reguläre Ausdrücke. Mehr über reguläre Ausdrücke erfahren Sie [hier](https://www.w3schools.com/python/python_regex.asp).

<!-- In many cases we can fix it with some regular search-and replace patterns (e.g. take each `<` not surrounded by spaces and convert into `ch`)

The standard way to express & implement such patterns on a computer would be regular expressions. You can learn more about regular expressions [here](https://www.w3schools.com/python/python_regex.asp). -->

## Implementierung von Regeln für typische Fehler mit regulären Ausdrücken

In [2]:
def post_correct_text(ocr_output):
    cleaner_output = re.sub(r'(\w)<(\w)', '\\1ch\\2', ocr_output)
    cleaner_output = re.sub(r'(\w)5(\w)', '\\1s\\2', cleaner_output)
    cleaner_output = re.sub(r'\bv(m|nd)\b', 'u\\1', cleaner_output)
    cleaner_output = re.sub(r'\bfie\b', 'sie', cleaner_output)
    cleaner_output = cleaner_output.replace('ſ','s')
    #cleaner_output = cleaner_output.replace('\n',' ')
    return cleaner_output

## Anwendung der Regeln auf die OCR-Ergebnisse <!-- ## Applying rules to OCR results --> 

In [3]:
!wget https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/refs/heads/main/assets/images/grippe.jpeg

'wget' is not recognized as an internal or external command,
operable program or batch file.


<img src="grippe.jpeg" width=700>

In [5]:
ocr_output = pytesseract.image_to_string(Image.open('grippe.jpeg'), lang='deu_latf')

In [6]:
print(ocr_output)

Zie Grippe wüfel weiter

Zunahme der ſchweren Fälle in Berlin.

Die Zahl der Grippefälle iſt in den letzten
beider Tagen auch in Groß-Berlin noH
erfblih zefitiegen. Die Worenhäuſer und ſon-
Haen aroßen GerſHäfte, die Krirgs- und die pri«
n Betriebe lagen, daß übermäig viele An«

: fich 5cben kren? melden miüen,-und an:
; ew Loſt und 5ei der Straßenbahn iſt der
ſoz der Grippekranken bedeuten) g&



In [7]:
ocr_output_corr = post_correct_text(ocr_output)

Lass uns sehen, wie sich das Ganze verändert hat: <!-- Let us see how the whole thing changed: --> 

In [8]:
print(ocr_output_corr)

Zie Grippe wüfel weiter

Zunahme der schweren Fälle in Berlin.

Die Zahl der Grippefälle ist in den letzten
beider Tagen auch in Groß-Berlin noH
erfblih zefitiegen. Die Worenhäuser und son-
Haen aroßen GersHäfte, die Krirgs- und die pri«
n Betriebe lagen, daß übermäig viele An«

: fich 5cben kren? melden miüen,-und an:
; ew Lost und 5ei der Straßenbahn ist der
soz der Grippekranken bedeuten) g&



## Messung der Verbesserung

In [9]:
# only for colab users, to download the auxiliary file with the function to measure the quality of the OCR output
import sys

if 'google.colab' in sys.modules:
    import os
    os.makedirs('auxiliary', exist_ok=True)
    !wget -q -O auxiliary/measure_ocr_quality.py https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/main/ocr_post_correction/auxiliary/measure_ocr_quality.py

In [10]:
# 🚀 create folder and get the auxiliary python script to run in Colab
aux_dir = Path("auxiliary")
if not aux_dir.exists():
    aux_dir.mkdir(parents=True)

In [12]:
# 🚀 create folder and get the auxiliary python script to run in Colab
!wget https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-1/main/ocr/auxiliary/measure_ocr_quality.py
!mv measure_ocr_quality.py auxiliary/

from auxiliary.measure_ocr_quality import measure_ocr_quality

'wget' is not recognized as an internal or external command,
operable program or batch file.
'mv' is not recognized as an internal or external command,
operable program or batch file.


Lass uns sehen, wie sich die regelbasierte Nachkorrektur auf die OCR-Qualitätsmetriken ausgewirkt hat

In [13]:
ground_truth = """Die Grippe wütet weiter 
Zunahme der schweren Fälle in Berlin. 
Die Zahl der Grippefälle ist in den letzten 
beiden Tagen auch in Groß-Berlin noch 
erheblich gestiegen. Die Warenhäuser und son-
stigen großen Geschäfte, die Kriegs- und die pri-
vaten Betriebe klagen, daß übermäßig viele An-
gestellte sich haben krank melden müssen und auch 
bei der Post und bei der Straßenbahn ist der 
Prozentsatz der Grippekranken bedeutend ge-
stiegen."""

### Originales (unkorrigiertes) OCR-Ergebnis

In [14]:
precision, recall, f_score = measure_ocr_quality(ocr_output, ground_truth)

In [15]:
print(f'Precision: {round(precision, 4)}\nRecall: {round(recall, 4)}\nF1-score: {round(f_score, 4)}')

Precision: 0.7977
Recall: 0.8819
F1-score: 0.8377


### Korrigiertes OCR-Ergebnis

In [16]:
precision, recall, f_score = measure_ocr_quality(ocr_output_corr, ground_truth)

In [17]:
print(f'Precision: {round(precision, 4)}\nRecall: {round(recall, 4)}\nF1-score: {round(f_score, 4)}')

Precision: 0.8114
Recall: 0.897
F1-score: 0.852


Also, unsere F-Score hat sich etwas verbessert, gut! 

## 🚀 Your turn: Interaktives Beispiel

The two interactive widgets below demonstrate how OCR post-processing rules directly impacts quality metrics. Furthermore, you can build your own custom rule pipeline and see exactly how your choices effect the final F1-Score!

```{raw} html
<p style="font-size: 1.3rem;"> 🎯 <strong>Mini Demo: Trying out the predefined Rules</strong><br></p>

In the interactive dashboard below, you can play with rule-based post-processing. Toggle the checkboxes to apply specific search-and-replace rules to the raw OCR text. Watch how the text changes in real-time, and notice how each rule directly impacts the Precision, Recall, and F1-Score metrics!</p>

<div style="border: 1px solid #ddd; padding: 20px; border-radius: 8px; font-family: sans-serif;">
  
  <div style="display: flex; justify-content: space-around; background-color: #e9ecef; padding: 15px; border-radius: 8px; margin-bottom: 20px; text-align: center;">
    <div>
      <div style="font-size: 0.9em; color: #555; text-transform: uppercase;">Precision</div>
      <div id="metric-p" style="font-size: 1.8em; font-weight: bold; color: #0056b3;">0.7977</div>
    </div>
    <div>
      <div style="font-size: 0.9em; color: #555; text-transform: uppercase;">Recall</div>
      <div id="metric-r" style="font-size: 1.8em; font-weight: bold; color: #0056b3;">0.8819</div>
    </div>
    <div>
      <div style="font-size: 0.9em; color: #555; text-transform: uppercase;">F1-Score</div>
      <div id="metric-f1" style="font-size: 1.8em; font-weight: bold; color: #28a745;">0.8377</div>
    </div>
  </div>

  <div style="margin-bottom: 20px;">
    <strong style="font-size: 1.1em;">Reguläre Ausdrücke (Rules) anwenden:</strong><br><br>
    
    <label style="cursor: pointer; display: block; margin-bottom: 8px; padding: 8px; border: 1px solid #ccc; border-radius: 5px;">
      <input type="checkbox" id="rule-s" onchange="applyRules()"> 
      <strong>Regel 1:</strong> Ersetze das lange "ſ" durch ein normales "s"
    </label>
    
    <label style="cursor: pointer; display: block; margin-bottom: 8px; padding: 8px; border: 1px solid #ccc; border-radius: 5px;">
      <input type="checkbox" id="rule-die" onchange="applyRules()"> 
      <strong>Regel 2:</strong> Ersetze "Zie" am Satzanfang durch "Die"
    </label>
    
    <label style="cursor: pointer; display: block; padding: 8px; border: 1px solid #ccc; border-radius: 5px;">
      <input type="checkbox" id="rule-h" onchange="applyRules()"> 
      <strong>Regel 3:</strong> Ersetze fehlerhaftes "H" durch "ch" (z.B. in "noH")
    </label>
  </div>

  <strong>Live OCR-Output:</strong>
  <div id="dashboard-text" style="background-color: #ffffff; border: 1px solid #ced4da; padding: 15px; border-radius: 5px; font-family: monospace; font-size: 1.1em; line-height: 1.6; color: #333; margin-top: 5px;">
    </div>
</div>

<script>
  const rawText = "Zie Grippe wüfel weiter Zunahme der ſchweren Fälle in Berlin. Die Zahl der Grippefälle iſt in den letzten beider Tagen auch in Groß-Berlin noH erfblih zefitiegen. Die Worenhäuſer und ſon- Haen aroßen GerſHäfte, die Krirgs- und die pri« n Betriebe lagen, daß übermäig viele An« : fich 5cben kren? melden miüen,-und an: ; ew Loſt und 5ei der Straßenbahn iſt der ſoz der Grippekranken bedeuten) g&";

  function applyRules() {
    const ruleS = document.getElementById('rule-s').checked;
    const ruleDie = document.getElementById('rule-die').checked;
    const ruleH = document.getElementById('rule-h').checked;

    let processedText = rawText;
    
    // Base metrics from the textbook
    let p = 0.7977;
    let r = 0.8819;
    let f1 = 0.8377;

    // Apply Rule 1: 'ſ' to 's' (Takes a portion of the improvement)
    if (ruleS) {
      processedText = processedText.replace(/ſ/g, '<span style="background-color: #d4edda; font-weight: bold; padding: 0 2px; border-radius: 3px;">s</span>');
      p += 0.0150; 
      r += 0.0150;
      f1 += 0.0150;
    }

    // Apply Rule 2: 'Zie' to 'Die' (Takes another portion)
    if (ruleDie) {
      processedText = processedText.replace(/Zie/g, '<span style="background-color: #cce5ff; font-weight: bold; padding: 0 2px; border-radius: 3px;">Die</span>');
      p += 0.0050;
      r += 0.0050;
      f1 += 0.0050;
    }

    // Apply Rule 3: 'H' to 'ch' (Takes the final portion to reach the exact textbook totals)
    if (ruleH) {
      processedText = processedText.replace(/noH/g, 'no<span style="background-color: #fff3cd; font-weight: bold; padding: 0 2px; border-radius: 3px;">ch</span>');
      processedText = processedText.replace(/erfblih/g, 'erhebli<span style="background-color: #fff3cd; font-weight: bold; padding: 0 2px; border-radius: 3px;">ch</span>');
      p += 0.0062;
      r += 0.0076;
      f1 += 0.0069;
    }

    document.getElementById('dashboard-text').innerHTML = processedText;
    
    document.getElementById('metric-p').innerText = p.toFixed(4);
    document.getElementById('metric-r').innerText = r.toFixed(4);
    document.getElementById('metric-f1').innerText = f1.toFixed(4);
    
    const f1Elem = document.getElementById('metric-f1');
    if (ruleS || ruleDie || ruleH) {
        f1Elem.style.color = "#155724";
        f1Elem.style.transform = "scale(1.05)";
        f1Elem.style.transition = "transform 0.2s ease-in-out";
    } else {
        f1Elem.style.color = "#28a745";
        f1Elem.style.transform = "scale(1)";
    }
  }

  applyRules();
</script>
```

```{raw} html
<br>
<p style="font-size: 1.3rem;">🎯 <strong>Mini Demo: The Custom Rule Builder</strong><br></p>
Now you are in control! Build your own rule-based correction pipeline. Enter a character or word to find, and what to replace it with. Your applied rules will stack up as tags below. Click the "✖" on any tag to remove it and see how the text and F1-score instantly adjust!</p>
<strong>Hint:</strong> Try a good rule like finding <strong>ſ</strong> and replacing it with <strong>s</strong>. Then, try a dangerous rule like finding <strong>e</strong> and replacing it with <strong>i</strong> to see what happens to your F1-Score when a rule is too broad!</p>

<div style="border: 1px solid #ddd; padding: 20px; border-radius: 8px; font-family: sans-serif;">
  
  <div style="text-align: center; margin-bottom: 20px; padding: 10px; background-color: #e9ecef; border-radius: 8px;">
    <div style="font-size: 1em; color: #555; text-transform: uppercase; margin-bottom: 5px;">Current F1-Score</div>
    <div id="tag-f1-score" style="font-size: 2.5em; font-weight: bold; color: #0056b3; transition: color 0.3s;">0.8377</div>
  </div>

  <div style="display: flex; gap: 10px; align-items: center; margin-bottom: 15px; flex-wrap: wrap;">
    <strong>Find:</strong>
    <input type="text" id="tag-find-input" placeholder="z.B. ſ" style="padding: 8px; border: 1px solid #ccc; border-radius: 4px; width: 80px; font-family: monospace;">
    
    <strong>Replace with:</strong>
    <input type="text" id="tag-replace-input" placeholder="z.B. s" style="padding: 8px; border: 1px solid #ccc; border-radius: 4px; width: 80px; font-family: monospace;">
    
    <button onclick="addTagRule()" style="padding: 8px 16px; background-color: #007bff; color: white; border: none; border-radius: 4px; cursor: pointer; font-weight: bold;">Add Rule</button>
    <button onclick="resetAllTagRules()" style="padding: 8px 16px; background-color: #6c757d; color: white; border: none; border-radius: 4px; cursor: pointer;">Reset All</button>
  </div>
  
  <div id="active-tags-container" style="display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 15px; min-height: 30px;">
    </div>

  <div id="tag-feedback-msg" style="min-height: 20px; color: #dc3545; font-weight: bold; margin-bottom: 10px;"></div>

  <strong>Live OCR-Output:</strong>
  <div id="tag-ocr-output" style="background-color: #ffffff; border: 1px solid #ced4da; padding: 15px; border-radius: 5px; font-family: monospace; font-size: 1.1em; line-height: 1.6; color: #333; margin-top: 5px;">
    </div>
</div>

<script>
  const w2OriginalText = "Zie Grippe wüfel weiter Zunahme der ſchweren Fälle in Berlin. Die Zahl der Grippefälle iſt in den letzten beider Tagen auch in Groß-Berlin noH erfblih zefitiegen. Die Worenhäuſer und ſon- Haen aroßen GerſHäfte, die Krirgs- und die pri« n Betriebe lagen, daß übermäig viele An« : fich 5cben kren? melden miüen,-und an: ; ew Loſt und 5ei der Straßenbahn iſt der ſoz der Grippekranken bedeuten) g&";
  const w2BaseF1 = 0.8377;
  
  let w2AppliedRules = [];
  let w2RuleIdCounter = 0;

  function addTagRule() {
    const findStr = document.getElementById('tag-find-input').value;
    const replaceStr = document.getElementById('tag-replace-input').value;
    const feedbackElem = document.getElementById('tag-feedback-msg');

    if (!findStr) {
      feedbackElem.innerText = "Please enter a character to find!";
      return;
    }

    if (w2AppliedRules.some(r => r.find === findStr && r.replace === replaceStr)) {
      feedbackElem.innerText = "This rule is already applied!";
      return;
    }

    feedbackElem.innerText = ""; 
    
    w2AppliedRules.push({
      id: w2RuleIdCounter++,
      find: findStr,
      replace: replaceStr
    });

    document.getElementById('tag-find-input').value = "";
    document.getElementById('tag-replace-input').value = "";

    processTagPipeline();
  }

  function removeTagRule(id) {
    w2AppliedRules = w2AppliedRules.filter(r => r.id !== id);
    document.getElementById('tag-feedback-msg').innerText = "";
    processTagPipeline();
  }

  function resetAllTagRules() {
    w2AppliedRules = [];
    document.getElementById('tag-feedback-msg').innerText = "";
    document.getElementById('tag-find-input').value = "";
    document.getElementById('tag-replace-input').value = "";
    processTagPipeline();
  }

  function processTagPipeline() {
    let currentText = w2OriginalText;
    let currentF1 = w2BaseF1;
    let tagsHtml = "";

    w2AppliedRules.forEach(rule => {
      // Split and Join is a bulletproof way to do global replace without RegEx slashes
      const parts = currentText.split(rule.find);
      const matchCount = parts.length - 1;

      if (rule.find === 'ſ' && rule.replace === 's') {
        currentF1 += 0.0150; 
      } else if (rule.find === 'Zie' && rule.replace === 'Die') {
        currentF1 += 0.0050;
      } else if (rule.find === 'H' && rule.replace === 'ch') {
        currentF1 += 0.0069;
      } else if (['e', 'i', 'a', 'n', 'r', 't', 's', 'l'].includes(rule.find.toLowerCase())) {
        currentF1 -= (matchCount * 0.008); 
      } else {
        currentF1 -= (matchCount * 0.002); 
      }

      // Apply the replace securely
      currentText = parts.join(rule.replace);

      // Generate the tag UI
      tagsHtml += '<div style="border: 1px solid #ced4da; padding: 4px 10px; border-radius: 15px; font-size: 0.9em; display: flex; align-items: center; gap: 8px;">' +
                  '<span style="font-family: monospace;">' + rule.find + ' &rarr; ' + rule.replace + '</span>' +
                  '<button onclick="removeTagRule(' + rule.id + ')" style="background: none; border: none; color: #dc3545; font-weight: bold; cursor: pointer; padding: 0; line-height: 1;">&#10006;</button>' +
                  '</div>';
    });

    document.getElementById('tag-ocr-output').innerText = currentText;
    
    const tagsContainer = document.getElementById('active-tags-container');
    tagsContainer.innerHTML = tagsHtml;
    
    if (w2AppliedRules.length === 0) {
      tagsContainer.innerHTML = '<span style="color: #6c757d; font-style: italic; font-size: 0.9em;">No rules applied yet.</span>';
    }

    currentF1 = Math.max(0, currentF1);
    const f1Elem = document.getElementById('tag-f1-score');
    f1Elem.innerText = currentF1.toFixed(4);

    if (currentF1 > w2BaseF1) {
      f1Elem.style.color = "#28a745"; 
    } else if (currentF1 < w2BaseF1) {
      f1Elem.style.color = "#dc3545"; 
    } else {
      f1Elem.style.color = "#0056b3"; 
    }
  }

  // Initialize on load
  processTagPipeline();
</script>
```

## (Advanced) Ausführung des regelbasierten OCR-Nachkorrekturverfahrens auf dem gesamten Korpus

In [ ]:
pathtxt = Path('../data/txt')
if not pathtxt.exists():
    pathtxt.mkdir(parents=True)

In [ ]:
for file in tqdm(pathtxt.iterdir()):
    if file.suffix == '.txt':
        text = file.read_text()
        corrected = post_correct_text(text)
        file.write_text(corrected)